In [1]:
import numpy as np
import pandas as pd
from src.utils import load_activity
from src.featurization import MolecularFeaturizer, get_scaffolds, get_pic50, get_mols

c:\Users\elbec\anaconda3\envs\azure\Lib\site-packages\chembl_webresource_client\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


###LOADING LIGANDS FOR CHEMBL206

In [2]:
data = load_activity(
    target="CHEMBL206",
    assay_type="B",
    standard_type="IC50",
    standard_units="nM"
)
data

,canonical_smiles,standard_value,mol
0,B.CP(c1ccccc1)c1ccc(O)cc1,9800.00,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A6...
1,B.Oc1ccc(P(c2ccccc2)c2ccccc2)cc1,12000.00,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A6...
2,Br.C=CCOC(C)C1=CC(C)(C)Nc2ccc(-c3cc(F)ccc3OC)cc21,2000.00,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A6...
3,BrC1=CN2NC(c3cc4ccccc4n3Cc3ccccc3)=NC2N=C1,539.18,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A6...
4,Brc1ccc(-c2nnc(-c3cccnc3)c3conc23)cc1,6621.20,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A6...
...,...,...,...
3710,c1ccc(-c2[nH]c3ccccc3c2-c2ccccc2)cc1,6950.00,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A9...
3711,c1ccc(C2=C(c3ccc(OCCN4CCC4)cc3)c3ccccc3OCC2)cc1,8.77,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A9...
3712,c1ccc(Cn2c(-c3nc4ccccc4[nH]3)cc3ccccc32)cc1,259.16,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A9...
3713,c1ccc(OCc2nnc3sc(C4COc5ccccc5O4)nn23)cc1,9549.84,<rdkit.Chem.rdchem.Mol object at 0x0000026E8A9...


In [3]:
data.isna().value_counts()

canonical_smiles  standard_value  mol  
False             False           False    3715
Name: count, dtype: int64

In [ ]:
# Saving the data in raw
data.to_csv('../data/raw/chembl206.csv', index=False)

###FEATURIZATION

In [4]:
# Generation Descriptors & Morgan Fingerprints
featurizer = MolecularFeaturizer()
X = featurizer.transform(data["mol"])
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 3715 entries, 0 to 3714
Columns: 1036 entries, MolWt to FP_1024
dtypes: float32(1036)
memory usage: 14.7 MB


In [5]:
# Calcualting pIC50
y = get_pic50(data["standard_value"])
y.describe()

count    3715.000000
mean        6.732905
std         1.579754
min         2.301030
25%         5.303955
50%         6.772113
75%         8.000000
max        11.698970
Name: pIC50, dtype: float64

In [6]:
# Scaffold Generation
scaffolds = get_scaffolds(data["mol"])
scaffolds.info()

<class 'pandas.Series'>
RangeIndex: 3715 entries, 0 to 3714
Series name: scaffold
Non-Null Count  Dtype
--------------  -----
3715 non-null   str  
dtypes: str(1)
memory usage: 178.5 KB


In [7]:
df = pd.concat([X, y, scaffolds], axis=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3715 entries, 0 to 3714
Columns: 1038 entries, MolWt to scaffold
dtypes: float32(1036), float64(1), str(1)
memory usage: 14.9 MB


In [8]:
# Saving the processed data
df.to_csv('../data/processed/df.csv', index=False)

### ZINC DATA FEATURIZATION

In [56]:
zinc = pd.read_csv(
    '../data/raw/zinc_molecules.smi', 
    sep=r'\s+',  
    dtype=str,
    engine='c'
)
zinc = zinc.dropna(subset=["smiles"]).reset_index(drop=True)
zinc = zinc.drop_duplicates(subset=["smiles"]).reset_index(drop=True)
zinc.info()

<class 'pandas.DataFrame'>
RangeIndex: 2310596 entries, 0 to 2310595
Data columns (total 2 columns):
 #   Column  Dtype
---  ------  -----
 0   smiles  str  
 1   id      str  
dtypes: str(2)
memory usage: 150.5 MB


In [58]:
zinc["mol"] = get_mols(zinc["smiles"])

[03:55:09] Explicit valence for atom # 3 Sn, 5, is greater than permitted


ValueError: Invalid SMILES: CCC[SnH2+](CCC)CCC

In [ ]:
X_zinc = featurizer.transform(zinc["mol"])
X_zinc.info()

Traceback (most recent call last):
  File "c:\Users\elbec\anaconda3\envs\azure\Lib\site-packages\rdkit\ML\Descriptors\MoleculeDescriptors.py", line 88, in CalcDescriptors
    res[i] = fn(mol)
             ^^^^^^^
  File "c:\Users\elbec\anaconda3\envs\azure\Lib\site-packages\rdkit\Chem\Descriptors.py", line 75, in <lambda>
    MolWt = lambda *x, **y: _rdMolDescriptors._CalcMolWt(*x, **y)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Boost.Python.ArgumentError: Python argument types in
    rdkit.Chem.rdMolDescriptors._CalcMolWt(str)
did not match C++ signature:
    _CalcMolWt(class RDKit::ROMol mol, bool onlyHeavy=False)
Traceback (most recent call last):
  File "c:\Users\elbec\anaconda3\envs\azure\Lib\site-packages\rdkit\ML\Descriptors\MoleculeDescriptors.py", line 88, in CalcDescriptors
    res[i] = fn(mol)
             ^^^^^^^
  File "c:\Users\elbec\anaconda3\envs\azure\Lib\site-packages\rdkit\Chem\Lipinski.py", line 78, in HeavyAtomCount
    return mol.GetNumHeavy

ArgumentError: Python argument types in
    rdkit.Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(str)
did not match C++ signature:
    GetMorganFingerprintAsBitVect(class RDKit::ROMol mol, unsigned int radius, unsigned int nBits=2048, class boost::python::api::object invariants=[], class boost::python::api::object fromAtoms=[], bool useChirality=False, bool useBondTypes=True, bool useFeatures=False, class boost::python::api::object bitInfo=None, bool includeRedundantEnvironments=False)